In [9]:
import pandas as pd
import numpy
import json
import os
from dotenv import load_dotenv
from copy import deepcopy
load_dotenv()
os.chdir(os.getenv('PARENT_DIR'))

In [10]:
from typing import List, Dict
import re
def parse_absa_string(text: str) -> List[Dict[str, str]]:
    """
    Parses a string formatted as "[A] aspect [O] opinion [S] sentiment" into a list of dictionaries.
    Each dictionary contains the tag as the key and the corresponding value.
    For example, "[A] [O] [S] [A] harga [O] terjangkau [S] positive [SSEP] [A] fasilitas [O] nyaman [S] positive" becomes:
    [{'A': 'harga', 'S': 'positive', 'O': 'terjangkau'},
    {'A': 'fasilitas', 'S': 'positive', 'O': 'nyaman'}].

    Args:
        text (str): ABSA string output to be parsed.

    Returns:
        List[Dict[str, str]]: List of dictionaries of parsed ABSA output.

    """
    pattern = r"\[(\w+)\]\s*([^[]+)"
    matches = re.findall(pattern, text)

    result = []
    current_dict = {}

    for tag, content in matches:
        if tag == "SSEP":  # Sentence separator -> Start a new dictionary
            result.append(current_dict)
            current_dict = {}
        else:
            current_dict[tag] = content.strip()

    if current_dict:  # Append the last sentence if it exists
        result.append(current_dict)

    return result

def calculate_metrics(predictions: List[List[Dict[str, str]]], targets: List[List[Dict[str, str]]]) -> Dict[str, float]:
    """
    Calculate precision, recall, and F1 score for the given predictions and targets for ABSA.

    Args:
        predictions (List[List[Dict[str, str]]]): List of predicted triplets.
        targets (List[List[Dict[str, str]]]): List of target triplets.
        task (str): The task name for which metrics are calculated.
    
    Returns:
        Dict[str, float]: A dictionary containing precision, recall, and F1 score.
    """
    true_positive = 0
    false_positive = 0
    false_negative = 0
    for prediction,target in zip(predictions,targets):
        for target_tuple in target:
            if target_tuple in prediction:
                true_positive += 1
            else:
                false_negative += 1
        false_positive += sum(1 for pred in prediction if pred not in target)
    precision = true_positive/(true_positive + false_positive) if (true_positive + false_positive) > 0 else 0
    recall = true_positive/(true_positive + false_negative) if (true_positive + false_negative) > 0 else 0
    f1 = (2 * recall * precision)/(recall + precision) if (recall + precision) > 0 else 0
    return {
        f"precision" : precision,
        f"recall" : recall,
        f"f1" : f1
    }

def convert_absadict_to_tuples(absa_dict_list: List[Dict[str, str]]) -> List[tuple]:
	"""
	Convert a list of ABSA dictionaries to a list of tuples in the format (aspect, opinion, sentiment).

	Args:
		absa_dict_list (List[Dict[str, str]]): List of dictionaries containing ABSA information.

	Returns:
		List[tuple]: List of tuples in the format (aspect, opinion, sentiment).
	"""
	tuples_list = []
	for absa_dict in absa_dict_list:
		aspect = absa_dict.get('A', '')
		opinion = absa_dict.get('O', '')
		sentiment = absa_dict.get('S', '')
		tuples_list.append((aspect, opinion, sentiment))
	return tuples_list

def convert_tuples_to_absadict(tuples_list: List[tuple], order: str) -> List[Dict[str, str]]:
	"""
	Convert a list of tuples in the format (aspect, opinion, sentiment) to a list of ABSA dictionaries.

	Args:
		tuples_list (List[tuple]): List of tuples in the format (aspect, opinion, sentiment).
		order (str): The order of the output dictionary keys. It can be 'AOS', 'ASO', 'OAS', 'OSA', 'SAO', or 'SOA'.

	Returns:
		List[Dict[str, str]]: List of dictionaries containing ABSA information.
	"""
	absa_dict_list = []
	for aspect, opinion, sentiment in tuples_list:
		absa_dict = {}
		for char in order:
			if char == 'A':
				absa_dict['A'] = aspect
			elif char == 'O':
				absa_dict['O'] = opinion
			elif char == 'S':
				absa_dict['S'] = sentiment
			else:
				raise ValueError(f"Invalid character '{char}' in order string. Only 'A', 'O', and 'S' are allowed.")
		absa_dict_list.append(absa_dict)
	return absa_dict_list

def convert_output_to_mvp_format(data_list: List[Dict[str, str]], order='aos') -> str:
	"""
	Converts a list of aspect-based sentiment dictionaries to the 
	extraction-style string format used in the MVP paper.

	Args:
		data_list (List[Dict[str, str]]): A list of strings, where each dictionary must contain 
				the keys 'A' (Aspect), 'O' (Opinion), and 'S' (Sentiment).
		order (str): A string specifying the order of elements (e.g., 'aos', 'ao', 'as', 'a', 'o')

	Returns:
		A single string formatted with special tokens and elements based on the order
	"""

	result_str = []
	for triplet in data_list:
		triplet_str = [f"[{element.upper()}] {triplet[element.upper()]}" for element in order]
		triplet_str = ' '.join(triplet_str)
		result_str.append(triplet_str)
	return ' [SSEP] '.join(result_str)

def convert_output_to_mvp_nossep(data_list: List[Dict[str, str]], order='aos') -> str:
	"""
	Converts a list of aspect-based sentiment dictionaries to the 
	extraction-style string format used in the MVP paper.

	Args:
		data_list (List[Dict[str, str]]): A list of strings, where each dictionary must contain 
				the keys 'A' (Aspect), 'O' (Opinion), and 'S' (Sentiment).
		order (str): A string specifying the order of elements (e.g., 'aos', 'ao', 'as', 'a', 'o')

	Returns:
		A single string formatted with special tokens and elements based on the order
	"""

	result_str = []
	for triplet in data_list:
		triplet_str = [f"[{element.upper()}] {triplet[element.upper()]}" for element in order]
		triplet_str = ' '.join(triplet_str)
		result_str.append(triplet_str)
	return result_str

In [19]:
from glob import glob
data_paths = glob('outputs/evals/hotel_reviews/*/mvp/seed_*/*/*/*/*/inference_results.json')

In [20]:
data_results = {
    'lang': [],
    'seed': [],
    'use_constrained_decoding': [],
    'precision': [],
    'recall': [],
    'f1': []
}

In [21]:
data_paths[:2]

['outputs/evals/hotel_reviews/jav/mvp/seed_123/20260402_134023_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-7760/checkpoint-7760/unconstrained_decoding/inference_results.json',
 'outputs/evals/hotel_reviews/jav/mvp/seed_31415/20260402_152635_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-7760/checkpoint-7760/unconstrained_decoding/inference_results.json']

In [22]:
data_paths[0].split('/')

['outputs',
 'evals',
 'hotel_reviews',
 'jav',
 'mvp',
 'seed_123',
 '20260402_134023_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10',
 'checkpoint-7760',
 'checkpoint-7760',
 'unconstrained_decoding',
 'inference_results.json']

In [23]:
data_paths[0].split('/')[-1].split('\\')

['inference_results.json']

In [24]:
for data_path in data_paths:
	with open(data_path, 'r', encoding='utf-8') as f:
		data = json.load(f)
	lang = data_path.split('/')[3]
	# seed = data_path.split('/')[-1].split('\\')[1].split('_')[1]
	seed = data_path.split('/')[5].split('_')[1]
	# use_constrained_decoding = data_path.split('/')[-1].split('\\')[-2]
	use_constrained_decoding = data_path.split('/')[-1]
	if use_constrained_decoding == 'constrained_decoding':
		use_constrained_decoding = True
	else:		
		use_constrained_decoding = False 

	i = 0
	n_sample = 5
	selected_preds = []
	selected_targets = []
	table_freq = {}
	for idx, instance in enumerate(data):
		tuples = convert_absadict_to_tuples(parse_absa_string(instance['prediction']))
		for t in tuples:
			table_freq[t] = table_freq.get(t, 0) + 1
		i += 1
		if i >= n_sample:
			# Select majority
			majority_preds = [t for t, freq in table_freq.items() if freq > n_sample//2]
			selected_preds.append(majority_preds)

			# Append target
			target_tuples = convert_absadict_to_tuples(parse_absa_string(instance['target']))
			selected_targets.append(target_tuples)
			
			# Reset for next sample
			i = 0
			table_freq = {}
			# print(n_sample//2)
			# break
	
	metric_result = calculate_metrics(selected_preds, selected_targets)
	data_results['lang'].append(lang)
	data_results['seed'].append(seed)
	data_results['use_constrained_decoding'].append(use_constrained_decoding)
	data_results['precision'].append(metric_result['precision'])
	data_results['recall'].append(metric_result['recall'])
	data_results['f1'].append(metric_result['f1'])

	# Save the selected predictions and targets for the current data path
	folder_data_path = os.path.dirname(data_path)
	save_path = os.path.join(folder_data_path, 'voting_results.json')
	absa_str_selected_preds = [convert_output_to_mvp_format(convert_tuples_to_absadict(pred, order='AOS'), order='aos') for pred in selected_preds]
	absa_str_selected_targets = [convert_output_to_mvp_format(convert_tuples_to_absadict(target, order='AOS'), order='aos') for target in selected_targets]
	voting_results = []
	for i, pred in enumerate(selected_preds):
		voting_results.append(
			{
				'target': absa_str_selected_targets[i],
				'prediction': absa_str_selected_preds[i],
				'target_list': [f'[A] {target[0]} [O] {target[1]} [S] {target[2]}' for target in selected_targets[i]],
				'prediction_list': [f'[A] {p[0]} [O] {p[1]} [S] {p[2]}' for p in pred]
			}
		)
	with open(save_path, 'w', encoding='utf-8') as f:
		json.dump(voting_results, f, ensure_ascii=False, indent=4)

In [9]:
pd.DataFrame(data_results)

,lang,seed,use_constrained_decoding,precision,recall,f1
0,indo,123,True,0.708579,0.683188,0.695652
1,indo,123,False,0.707419,0.680174,0.693529
2,indo,2024,True,0.702274,0.682518,0.692255
3,indo,2024,False,0.699344,0.678500,0.688764
4,indo,31415,True,0.717301,0.694240,0.705582
5,indo,31415,False,0.715126,0.691896,0.703319
6,indo,777,True,0.720083,0.695244,0.707446
7,indo,777,False,0.716071,0.690891,0.703255
8,indo,9584,True,0.712858,0.692565,0.702565
9,indo,9584,False,0.711180,0.690221,0.700544
